In [1]:
import torch
import pickle
import chromadb
import pandas as pd

from pathlib import Path
from utils.funcs import * 
from tqdm.autonotebook import tqdm

/home/ytian/anaconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tcrs = pd.read_parquet("data/antigen_specific_tcrs.parquet")
print(tcrs.shape) 
tcrs.head()

(48540, 9)


,Species,Antigen Epitope,Antigen Protein,Antigen Source,CDR3.beta.aa,TRBV,TRBJ,Reference,Database
0,Mouse,IKAVYNFATCG,Pre-glycoprotein polyprotein GP complex,LCMV,CASSDAGANTEVF,TRBV8-1,TRBJ1-1,PMID:1716213,McPAS-TCR
1,Mouse,IKAVYNFATCG,Pre-glycoprotein polyprotein GP complex,LCMV,CASSDAGAYAEQF,TRBV8-1,TRBJ2-1,PMID:1716213,McPAS-TCR
2,Mouse,IKAVYNFATCG,Pre-glycoprotein polyprotein GP complex,LCMV,CASSDAGGAAEVF,TRBV8-3,TRBJ1-1,PMID:1716213,McPAS-TCR
3,Mouse,IKAVYNFATCG,Pre-glycoprotein polyprotein GP complex,LCMV,CASSDAGHSPLYF,TRBV8-1,TRBJ1-6,PMID:1716213,McPAS-TCR
4,Mouse,IKAVYNFATCG,Pre-glycoprotein polyprotein GP complex,LCMV,CASSDAWGGAEQYF,TRBV8-3,TRBJ2-6,PMID:1716213,McPAS-TCR


In [3]:
tcrs["CDR3.beta.aa"].str.len().max()

38

In [4]:
tcrs["CDR3.beta.aa"].duplicated().any()

False

In [5]:
tcrs["Species"].value_counts()

Species
Human    45750
Mouse     2790
Name: count, dtype: int64

In [6]:
tcrs["Antigen Source"].value_counts()[:5]

Antigen Source
CMV           19441
SARS-CoV-2     5000
Influenza      4226
EBV            4104
InfluenzaA     3585
Name: count, dtype: int64

In [7]:
# downsample CMV TCRs
tcrs_cmv = tcrs.loc[tcrs["Antigen Source"] == "CMV"]
tcrs_other = tcrs.loc[tcrs["Antigen Source"] != "CMV"]

tcrs_cmv = tcrs_cmv.sample(n=5000, random_state=0)
tcrs = pd.concat([tcrs_cmv, tcrs_other], ignore_index=True)
print(tcrs.shape)
tcrs.head()

(34099, 9)


,Species,Antigen Epitope,Antigen Protein,Antigen Source,CDR3.beta.aa,TRBV,TRBJ,Reference,Database
0,Human,KLGGALQAK,IE1,CMV,CASSPKTSVTYNEQFF,TRBV7-9*01,TRBJ2-1*01,https://www.10xgenomics.com/resources/applicat...,VDJdb
1,Human,NLVPMVATV,pp65,CMV,CASSLDSLNTIYF,TRBV5-1*01,TRBJ1-3*01,PMID:28423320,VDJdb
2,Human,KLGGALQAK,IE1,CMV,CASSSRTSSTDTQYF,TRBV12-4*01,TRBJ2-3*01,https://www.10xgenomics.com/resources/applicat...,VDJdb
3,Human,KLGGALQAK,IE1,CMV,CSSESGTSEAFF,TRBV29-1*01,TRBJ1-1*01,https://www.10xgenomics.com/resources/applicat...,VDJdb
4,Human,KLGGALQAK,IE1,CMV,CSVEYGLAGSTDTQYF,TRBV29-1*01,TRBJ2-3*01,https://www.10xgenomics.com/resources/applicat...,VDJdb


In [8]:
pubmed_base_url = "https://pubmed.ncbi.nlm.nih.gov/"

# pubmed link
def format_link(s):
    if s.startswith("PMID:"):
        return s.replace("PMID:", pubmed_base_url)
    return s

tcrs["Reference"] = tcrs["Reference"].map(format_link)
tcrs["Reference"].iloc[0]

'https://www.10xgenomics.com/resources/application-notes/a-new-way-of-exploring-immunity-linking-highly-multiplexed-antigen-recognition-to-immune-repertoire-and-phenotype/#'

In [9]:
tokenizer, model = get_model("facebook/esm2_t6_8M_UR50D")

2023-11-26 22:18:16.970 
  command:

    streamlit run /home/ytian/anaconda3/envs/ml/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
embeddings = []
batch_size = 128
seqs = tcrs["CDR3.beta.aa"].tolist()

for i in tqdm(range(0, len(seqs), batch_size)):
    embeds = get_embeddings(tokenizer, model, seqs[i:i+batch_size])
    embeddings.append(embeds)

embeddings = np.concatenate(embeddings, axis=0)
embeddings.shape

100%|██████████| 267/267 [00:12<00:00, 20.89it/s]


(34099, 320)

In [11]:
# convert to list
embeddings = embeddings.tolist()
len(embeddings)

34099

In [12]:
# use index as id
ids = [f"tcr{i}" for i in tcrs.index.tolist()]
ids[:5]

['tcr0', 'tcr1', 'tcr2', 'tcr3', 'tcr4']

In [13]:
# prepare metadata
metadatas = tcrs.to_dict("records")
metadatas[:1]

[{'Species': 'Human',
  'Antigen Epitope': 'KLGGALQAK',
  'Antigen Protein': 'IE1',
  'Antigen Source': 'CMV',
  'CDR3.beta.aa': 'CASSPKTSVTYNEQFF',
  'TRBV': 'TRBV7-9*01',
  'TRBJ': 'TRBJ2-1*01',
  'Reference': 'https://www.10xgenomics.com/resources/application-notes/a-new-way-of-exploring-immunity-linking-highly-multiplexed-antigen-recognition-to-immune-repertoire-and-phenotype/#',
  'Database': 'VDJdb'}]

In [14]:
# chroma
client = chromadb.PersistentClient(path="models")
collection = client.get_or_create_collection(
    name="tcrs", metadata={"hnsw:space": "cosine"}
)

In [15]:
collection.count()

0

In [16]:
upsert_to_collection(collection, ids, embeddings, metadatas, batch_size=10000)

100%|██████████| 4/4 [00:23<00:00,  5.82s/it]


In [17]:
collection.count()

34099

In [18]:
tcrs.to_parquet("data/antigen_specific_tcrs_subset.parquet")

with open('models/embeddings.pkl', 'wb') as f:
  pickle.dump(embeddings, f)